# 제3장. 구조적 사고: MECE와 로직 트리

## 학습 목표
1. MECE 원칙을 이해하고 실무 문제에 적용할 수 있다
2. Why Tree와 How Tree를 활용하여 원인 분석과 해결책 도출을 수행할 수 있다
3. 가설 기반 접근법과 피라미드 원칙으로 효율적 분석을 수행할 수 있다
4. AI를 문제 구조화의 보조 도구로 활용하되, 한계를 인식할 수 있다

---

### 강의 구조 (3시간)

| 시간 | 파트 | 내용 |
|------|------|------|
| | Part 1 | MECE 원칙 + 조사 과제 |
| | Part 2 | 로직 트리 (Why Tree, How Tree) 이론 |
| | 휴식 | |
| | Part 3 | 가설 기반 접근법과 피라미드 원칙 + 조사 과제 |
| | Part 4 | 종합 실습: 스타트업 성장 둔화 분석 |

---

In [89]:
# ============================================================
# 환경설정 및 라이브러리 로드
# ============================================================
# 처음 사용 시 아래 명령을 터미널에서 먼저 실행하세요:
#   python setup_env.py
# 그 후 VSCode 우측 상단에서 커널을 'AI 기획 강의 (Python 3)'으로 선택하세요.
# ============================================================

import sys

# 패키지 확인
_required = ["numpy", "pandas", "plotly"]
_missing = []
for _pkg in _required:
    try:
        __import__(_pkg)
    except ImportError:
        _missing.append(_pkg)
if _missing:
    print(f"❌ 누락된 패키지: {', '.join(_missing)}")
    print("   터미널에서 다음 명령을 실행하세요: python setup_env.py")
    raise SystemExit(1)

# 라이브러리 임포트
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import networkx as nx

import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print("✅ 라이브러리 로드 완료!")

✅ 라이브러리 로드 완료!


---

## Part 1. MECE 원칙

---

## 3.1 MECE란 무엇인가?

### 정의

**MECE** = **M**utually **E**xclusive, **C**ollectively **E**xhaustive

- **상호 배타적 (Mutually Exclusive)**: 각 항목이 서로 **겹치지** 않는다
- **전체 포괄적 (Collectively Exhaustive)**: 모든 항목을 합치면 전체를 **빠짐없이** 커버한다

### 기원

- 1960년대 후반, **McKinsey & Company**에서 **Barbara Minto**가 개발
- 이후 전략 컨설팅 업계 전반에서 문제 구조화의 **표준**으로 정착
- 아리스토텔레스의 논리학까지 거슬러 올라가는 지적 전통

---

### MECE vs Non-MECE 예시

| 분류 | MECE 여부 | 이유 |
|------|----------|------|
| 남성 / 여성 | ✅ MECE | 겹침 없음, 전체 포괄 |
| 청년 / 직장인 | ❌ Non-MECE | 젊은 직장인은 양쪽에 해당 (겹침) |
| 20대 / 30대 | ❌ Non-MECE | 40대 이상, 10대 이하 누락 |
| 출생연도별 분류 | ✅ MECE | 한 사람은 하나의 연도, 모든 사람 포괄 |
| 주사위 눈 (1~6) | ✅ MECE | 겹침 없음, 모든 결과 포괄 |

In [90]:
# 시각화: MECE vs Non-MECE
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('MECE Example', 'Non-MECE Example'),
    specs=[[{'type': 'pie'}, {'type': 'pie'}]]
)

# MECE: Revenue = Volume x Price
fig.add_trace(
    go.Pie(
        labels=['Volume (Sales Qty)', 'Price (Unit Price)'],
        values=[50, 50],
        marker_colors=['#4ECDC4', '#FF6B6B'],
        textinfo='label',
        hole=0.3
    ),
    row=1, col=1
)

# Non-MECE: Overlapping categories
fig.add_trace(
    go.Pie(
        labels=['Youth', 'Workers', 'Overlap (Young Workers)', 'Missing (Others)'],
        values=[25, 25, 15, 35],
        marker_colors=['#45B7D1', '#96CEB4', '#FF0000', '#CCCCCC'],
        textinfo='label',
        hole=0.3
    ),
    row=1, col=2
)

fig.update_layout(
    title_text='MECE vs Non-MECE Classification',
    height=400,
    showlegend=False
)

fig.show()

print("💡 왼쪽(MECE): 매출 = 판매량 × 단가 → 겹침 없이 전체를 커버")
print("💡 오른쪽(Non-MECE): '청년'과 '직장인' → 겹침(빨간색) + 누락(회색) 존재")

💡 왼쪽(MECE): 매출 = 판매량 × 단가 → 겹침 없이 전체를 커버
💡 오른쪽(Non-MECE): '청년'과 '직장인' → 겹침(빨간색) + 누락(회색) 존재


### MECE의 가치와 한계

| 구분 | 가치 | 한계 |
|------|------|------|
| 논리성 | 분석의 완결성 보장 | 현실 문제의 모호한 경계 반영 어려움 |
| 커뮤니케이션 | 명확하고 설득력 있는 구조 | 과도한 단순화로 뉘앙스 손실 가능 |
| 업무 관리 | 역할 분담 명확, 중복 방지 | 범주 간 협업 필요 시 경직성 |
| 창의성 | 체계적 사고 촉진 | 범주를 넘는 통합적 시각 억제 가능 |
| 효율성 | 분석 범위 명확화 | 완벽한 MECE 추구 시 과도한 시간 소요 |

> **핵심:** MECE는 엄격한 규칙이 아니라 **사고의 품질을 높이는 지침**이다.
> 80/20 원칙에 따라 핵심 요소에 집중하는 것이 실무적으로 더 효과적인 경우가 많다.

---

### MECE 분류 프레임워크

| 프레임워크 | 분류 기준 | 적합한 상황 | 예시 |
|------------|----------|-------------|------|
| **시간 기반** | 과거/현재/미래 | 전략 수립, 로드맵 작성 | 1년/3년/5년 목표 |
| **주체 기반** | 내부/외부, 3C | 환경 분석, 이해관계자 분석 | 고객/경쟁/자사 |
| **프로세스 기반** | 투입/과정/산출 | 운영 개선, 프로젝트 관리 | R&D/생산/판매 |
| **이분법** | 정량/정성, 통제가능/불가능 | 우선순위 결정, 리스크 분류 | 내부 요인/외부 요인 |

In [91]:
# 시각화: MECE 분류 프레임워크 적용 사례
frameworks = {
    'Time-based': ['Past', 'Present', 'Future'],
    'Entity-based': ['Customer', 'Competitor', 'Company'],
    'Process-based': ['Input', 'Process', 'Output'],
    'Dichotomy': ['Internal', 'External']
}

colors = {
    'Time-based': ['#FFB3BA', '#BAFFC9', '#BAE1FF'],
    'Entity-based': ['#FF6B6B', '#4ECDC4', '#45B7D1'],
    'Process-based': ['#96CEB4', '#FFEAA7', '#DDA0DD'],
    'Dichotomy': ['#FF9F43', '#54A0FF']
}

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=list(frameworks.keys()),
    specs=[[{'type': 'pie'}, {'type': 'pie'}],
           [{'type': 'pie'}, {'type': 'pie'}]]
)

positions = [(1,1), (1,2), (2,1), (2,2)]
for (name, labels), (r, c) in zip(frameworks.items(), positions):
    fig.add_trace(
        go.Pie(
            labels=labels,
            values=[1]*len(labels),
            marker_colors=colors[name],
            textinfo='label',
            hole=0.3
        ),
        row=r, col=c
    )

fig.update_layout(
    title_text='MECE Classification Frameworks',
    height=600,
    showlegend=False
)
fig.show()

print("💡 각 프레임워크는 겹침 없이(ME) 전체를 커버(CE)하는 분류 체계")
print("💡 문제 유형에 따라 적절한 프레임워크를 선택하는 것이 핵심")

💡 각 프레임워크는 겹침 없이(ME) 전체를 커버(CE)하는 분류 체계
💡 문제 유형에 따라 적절한 프레임워크를 선택하는 것이 핵심


### 산업별 MECE 분류 사례

| 산업 | 분석 대상 | MECE 프레임워크 | 분해 요소 |
|------|----------|-----------------|----------|
| 제조업 | 품질 문제 | 4M | Man, Machine, Material, Method |
| 유통업 | 매출 | 객단가 x 객수 | 구매품목수 x 품목단가, 신규객 x 재방문객 |
| 금융업 | 수익성 | DuPont 분석 | 순이익률, 자산회전율, 재무레버리지 |
| IT서비스 | 고객생애가치 | AARRR | 획득, 활성화, 유지, 수익, 추천 |

---

### 📝 이론 과제 3-1 (15분)

#### 과제: MECE 원칙 이해하기

**질문:**

1. **MECE의 두 가지 조건**(상호 배타적, 전체 포괄적)을 자신의 말로 설명하시오. (2-3문장)

2. 다음 분류가 MECE인지 판단하고, 그렇지 않다면 **어떤 조건이 위반**되는지 설명하시오:
   - 고객을 "충성 고객", "신규 고객", "온라인 고객"으로 분류
   - 비용을 "고정비", "변동비"로 분류

3. **DuPont 분석**(ROE = 순이익률 x 자산회전율 x 재무레버리지)이 왜 MECE인지 설명하시오.

**조사 키워드:**
- "MECE principle McKinsey"
- "DuPont analysis decomposition"

**제출:** 아래 마크다운 셀에 **직접 타이핑** (복사 붙여넣기 금지)

---

### ✅ 과제 3-1 제출란

**학번:** ___________  
**이름:** ___________

#### 1. MECE의 두 가지 조건 설명

(여기에 작성)

#### 2. 분류의 MECE 여부 판단

**"충성 고객 / 신규 고객 / 온라인 고객":**

(여기에 작성)

**"고정비 / 변동비":**

(여기에 작성)

#### 3. DuPont 분석이 MECE인 이유

(여기에 작성)

**출처:** (URL)

---

---

## Part 2. 로직 트리

---

## 3.2 로직 트리(Logic Tree)

### 로직 트리란?

**로직 트리**는 문제를 시각적으로 분해하는 도구이다.

- **이슈 트리(Issue Tree)**라고도 불린다
- 핵심 질문을 최상위에 두고, 하위 질문으로 분해하는 피라미드 구조
- 각 분기는 **MECE 원칙**을 충족해야 한다

### 두 가지 유형

| 유형 | 질문 | 목적 | 방향 |
|------|------|------|------|
| **Why Tree** (진단) | "왜?" | 문제의 근본 원인 식별 | 과거 지향적 |
| **How Tree** (솔루션) | "어떻게?" | 해결책 도출 | 미래 지향적 |

---

### Why Tree 예시: 매출 하락 원인 분석

```
왜 매출이 하락했는가?
├─ 판매량 감소
│   ├─ 신규 고객 감소
│   │   ├─ 마케팅 효과 저하
│   │   ├─ 경쟁사 진입
│   │   └─ 제품 매력도 저하
│   └─ 기존 고객 이탈
│       ├─ 서비스 불만족
│       └─ 가격 경쟁력 상실
└─ 평균 단가 하락
    ├─ 할인 증가
    └─ 저가 제품 비중 증가
```

**핵심:** `매출 = 판매량 x 단가` → 수학적으로 MECE가 보장되는 분해!

### Toyota의 5 Whys

> Taiichi Ohno: 표면적 증상에 머무르지 않고 **"왜 그런가?"를 반복**함으로써 근본 원인에 도달할 수 있다.

### How Tree 예시: 시장 점유율 확대 전략

```
어떻게 시장 점유율을 확대할 것인가?
├─ 시장 확대
│   ├─ 신규 세그먼트 진출
│   │   ├─ B2B 시장 진출
│   │   └─ 프리미엄 라인 출시
│   └─ 지역 확장
│       ├─ 동남아 시장 진출
│       └─ 온라인 직판 강화
└─ 경쟁사 점유율 탈취
    ├─ 가격 경쟁력 강화
    ├─ 제품 차별화
    └─ 유통 채널 확대
```

### 적정 깊이 가이드

| 기준 | 설명 |
|------|------|
| 실행 가능성 | 리프 노드가 구체적인 액션으로 전환 가능한가? |
| 분석 목적 | 진단용? → 검증 가능한 가설까지. 전략용? → 자원 배분 가능 수준까지 |
| 가용 시간 | 일반적으로 **3-4단계**가 적절 |

> **80/20 원칙:** 전체 문제의 80%를 설명하는 20%의 핵심 요인에 집중한다.

---

### 📝 이론 과제 3-2 (15분)

#### 과제: 로직 트리와 5 Whys

**질문:**

1. **Toyota의 5 Whys 기법**을 조사하고, "기계가 멈췄다"는 문제에 대해 5단계의 Why를 적용한 사례를 설명하시오. (3-4문장)

2. Why Tree와 How Tree의 **핵심 차이**를 한 문장으로 요약하시오.

3. "온라인 쇼핑몰의 장바구니 이탈률이 70%로 업계 평균(60%)보다 높다." 이 문제에 대해 **2단계 깊이의 Why Tree**를 작성하시오.

**조사 키워드:**
- "Toyota 5 Whys example"
- "cart abandonment rate causes"

**제출:** 아래 마크다운 셀에 **직접 타이핑** (복사 붙여넣기 금지)

---

### ✅ 과제 3-2 제출란

**학번:** ___________  
**이름:** ___________

#### 1. Toyota 5 Whys 사례

(여기에 작성)

#### 2. Why Tree vs How Tree 핵심 차이

(여기에 작성)

#### 3. 장바구니 이탈률 Why Tree

```
왜 장바구니 이탈률이 높은가?
├─ 
│   ├─ 
│   └─ 
└─ 
    ├─ 
    └─ 
```

**출처:** (URL)

---
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "AI 기획 강의 (Python 3)",
   "language": "python",
   "name": "strategy-lecture"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.14.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 2
}


---

## 3.3 가설 기반 접근법

### 가설 기반 사고(Hypothesis-Driven Thinking)

**답을 먼저 가정하고**, 사실 발견을 통해 **증명하거나 반증**하는 문제 해결 방법

> McKinsey 출신 컨설턴트: "가설이라는 단어는 McKinsey에서 가장 많이 사용되는 상위 5개 단어 중 하나"

### 귀납적 접근 vs 가설 기반 접근

| | 귀납적 접근 | 가설 기반 접근 |
|--|-----------|-------------|
| 순서 | 데이터 수집 → 패턴 발견 → 결론 | **결론 가정 → 검증 설계 → 데이터 확인** |
| 장점 | 개방적, 예상 밖 발견 가능 | **효율적**, 초점이 명확 |
| 단점 | 시간 소요 과다, 방향 잃기 쉬움 | 확증 편향 위험 |
| 비유 | 바다에서 그물 던지기 | **목표 물고기를 정해놓고 낚시** |

### 좋은 가설의 3가지 조건

| 조건 | 나쁜 예시 | 좋은 예시 |
|------|----------|----------|
| **구체성** | 시장이 성장할 것이다 | 국내 SaaS 시장이 연평균 15% 성장할 것이다 |
| **검증가능성** | 고객이 브랜드를 좋아한다 | NPS 점수가 50점 이상일 것이다 |
| **반증가능성** | 언젠가는 매출이 오를 것이다 | 다음 분기 매출이 10% 증가할 것이다 |

> **80/20 답:** 100% 확실성을 추구하기보다, **80% 확신에 도달하면 의사결정**을 내리고 다음 단계로!

---